In [ ]:
# ============================================================
# MOCAP FEATURE EXTRACTION — Full Feature Set
# Features: knee velocity, CoM displacement,
#           limb acceleration, trunk sway amplitude
# Output: mocap_features.csv saved to Google Drive
# ============================================================

import numpy as np
import pandas as pd
import os

# ── 1. Paths ──────────────────────────────────────────────────────────────
dataset_path = (
    '/content/501Project_Dataset/'
    'multimodal-synchronized-motion-capture-force-plate-and-radar-'
    'dataset-of-the-one-legged-stand-test-for-fall-risk-assessment-1.0'
)
output_path = '/content/drive/MyDrive/mocap_features.csv'

# ── 2. All 18 marker names (from PhysioNet table) ─────────────────────────
ALL_MARKERS = [
    'Ankle_R', 'Ankle_L', 'Knee_R', 'Knee_L',
    'Hip_R_Ant', 'Hip_L_Ant', 'Hip_R_Post', 'Hip_L_Post',
    'Wrist_R', 'Wrist_L', 'Elbow_R', 'Elbow_L',
    'Shoulder_R', 'Shoulder_L',
    'Belly', 'Chest', 'Lower_Back', 'Upper_Back'
]
TRUNK_MARKERS = ['Upper_Back', 'Lower_Back', 'Chest']

# ── 3. Load OLST attempts ─────────────────────────────────────────────────
olst_df = pd.read_csv(f'{dataset_path}/Metadata/OLST_Attempts.csv')
olst_df['participant_id'] = olst_df['OLST_attempt_id'].str.slice(0, 2).astype(int)
olst_df['movement_code']  = olst_df['RADAR_capture'].str.split('_').str[1]
print(f'✓ Loaded OLST_Attempts.csv — {len(olst_df)} attempts')

# ── 4. Feature functions ──────────────────────────────────────────────────

def knee_velocity_features(window, leg, prefix):
    """3D knee velocity: mean, std, max."""
    cols = [f'Knee_{leg}_pos_X', f'Knee_{leg}_pos_Y', f'Knee_{leg}_pos_Z']
    if not all(c in window.columns for c in cols):
        return {}
    vel = np.sqrt(
        window[cols[0]].diff()**2 +
        window[cols[1]].diff()**2 +
        window[cols[2]].diff()**2
    )
    return {
        f'{prefix}_knee_vel_mean': vel.mean(),
        f'{prefix}_knee_vel_std':  vel.std(),
        f'{prefix}_knee_vel_max':  vel.max(),
    }


def knee_acceleration_features(window, leg, prefix):
    """3D knee acceleration (2nd derivative of position): mean, std, max."""
    cols = [f'Knee_{leg}_pos_X', f'Knee_{leg}_pos_Y', f'Knee_{leg}_pos_Z']
    if not all(c in window.columns for c in cols):
        return {}
    # velocity then differentiate again for acceleration
    vel_x = window[cols[0]].diff()
    vel_y = window[cols[1]].diff()
    vel_z = window[cols[2]].diff()
    acc = np.sqrt(vel_x.diff()**2 + vel_y.diff()**2 + vel_z.diff()**2)
    return {
        f'{prefix}_knee_acc_mean': acc.mean(),
        f'{prefix}_knee_acc_std':  acc.std(),
        f'{prefix}_knee_acc_max':  acc.max(),
    }


def com_displacement_features(window):
    """
    Center of Mass approximation: mean position of all available markers.
    Displacement = how far CoM moves from its mean position over the window.
    """
    x_cols = [f'{m}_pos_X' for m in ALL_MARKERS if f'{m}_pos_X' in window.columns]
    y_cols = [f'{m}_pos_Y' for m in ALL_MARKERS if f'{m}_pos_Y' in window.columns]
    z_cols = [f'{m}_pos_Z' for m in ALL_MARKERS if f'{m}_pos_Z' in window.columns]

    if len(x_cols) == 0:
        return {}

    # CoM position per frame = mean across all markers
    com_x = window[x_cols].mean(axis=1)
    com_y = window[y_cols].mean(axis=1)
    com_z = window[z_cols].mean(axis=1)

    # Displacement from mean CoM position
    disp = np.sqrt(
        (com_x - com_x.mean())**2 +
        (com_y - com_y.mean())**2 +
        (com_z - com_z.mean())**2
    )

    # CoM velocity
    com_vel = np.sqrt(com_x.diff()**2 + com_y.diff()**2 + com_z.diff()**2)

    return {
        'com_disp_mean':     disp.mean(),
        'com_disp_std':      disp.std(),
        'com_disp_max':      disp.max(),
        'com_range_x':       com_x.max() - com_x.min(),  # mediolateral range
        'com_range_y':       com_y.max() - com_y.min(),  # anteroposterior range
        'com_vel_mean':      com_vel.mean(),
        'com_vel_std':       com_vel.std(),
    }


def trunk_sway_features(window):
    """
    Trunk sway amplitude using Upper_Back, Lower_Back, Chest markers.
    Sway = range and std of trunk marker positions in X (mediolateral)
    and Y (anteroposterior) axes.
    """
    available = [m for m in TRUNK_MARKERS if f'{m}_pos_X' in window.columns]
    if len(available) == 0:
        return {}

    # Average trunk position per frame
    trunk_x = window[[f'{m}_pos_X' for m in available]].mean(axis=1)
    trunk_y = window[[f'{m}_pos_Y' for m in available]].mean(axis=1)

    # Sway velocity
    sway_vel = np.sqrt(trunk_x.diff()**2 + trunk_y.diff()**2)

    return {
        'trunk_sway_x_std':      trunk_x.std(),    # mediolateral sway
        'trunk_sway_y_std':      trunk_y.std(),    # anteroposterior sway
        'trunk_sway_x_range':    trunk_x.max() - trunk_x.min(),
        'trunk_sway_y_range':    trunk_y.max() - trunk_y.min(),
        'trunk_sway_vel_mean':   sway_vel.mean(),
        'trunk_sway_vel_std':    sway_vel.std(),
    }


def extract_all_features(window, stance, lifted, attempt_id,
                          participant_id, label):
    """Combine all feature groups into one row."""
    if len(window) < 10:
        return None

    feats = {
        'OLST_attempt_id': attempt_id,
        'participant_id':  participant_id,
        'stance_leg':      stance,
        'lifted_leg':      lifted,
        'label':           label,
    }

    # Knee velocity — both legs
    feats.update(knee_velocity_features(window, stance, 'stance'))
    feats.update(knee_velocity_features(window, lifted, 'lifted'))

    # Knee acceleration — both legs
    feats.update(knee_acceleration_features(window, stance, 'stance'))
    feats.update(knee_acceleration_features(window, lifted, 'lifted'))

    # Center of Mass displacement
    feats.update(com_displacement_features(window))

    # Trunk sway amplitude
    feats.update(trunk_sway_features(window))

    return feats


# ── 5. Main extraction loop ───────────────────────────────────────────────
all_features = []
skipped      = 0

for idx, row in olst_df.iterrows():

    if (idx + 1) % 100 == 0:
        print(f'  Processing {idx+1}/{len(olst_df)}...')

    attempt_id     = row['OLST_attempt_id']
    participant_id = row['participant_id']
    radar_capture  = row['RADAR_capture']
    movement_code  = row['movement_code']
    t_foot_up      = row['t_foot_up']
    t_stable       = row['t_stable']
    t_break        = row['t_break']
    t_end          = row['t_end']

    stance = movement_code[-1]
    lifted = 'R' if stance == 'L' else 'L'

    # Load MOCAP file
    mocap_filename = radar_capture.replace('_RR_', '_MC_') + '_pos'
    mocap_file = f'{dataset_path}/Raw/MOCAP/{participant_id:02d}/{mocap_filename}.csv'
    if not os.path.exists(mocap_file):
        skipped += 1
        continue
    try:
        mocap = pd.read_csv(mocap_file)
    except Exception:
        skipped += 1
        continue

    # ── STABLE: t_stable → t_break (or t_end) ────────────────────
    if pd.notna(t_stable):
        t_stop = t_break if pd.notna(t_break) else t_end
        window = mocap[(mocap['time'] >= t_stable) & (mocap['time'] <= t_stop)]
        feat   = extract_all_features(window, stance, lifted,
                                      attempt_id, participant_id,
                                      label='STABLE')
        if feat:
            all_features.append(feat)

    # ── UNSTABLE: failed attempts only (t_stable is NaN) ──────────
    if pd.isna(t_stable):
        window = mocap[(mocap['time'] >= t_foot_up) & (mocap['time'] <= t_end)]
        feat   = extract_all_features(window, stance, lifted,
                                      attempt_id, participant_id,
                                      label='UNSTABLE')
        if feat:
            all_features.append(feat)

# ── 6. Save output ────────────────────────────────────────────────────────
mocap_features_df = pd.DataFrame(all_features)

print(f'\n✓ Extraction complete!')
print(f'  Total rows  : {len(mocap_features_df)}')
print(f'  STABLE      : {(mocap_features_df["label"] == "STABLE").sum()}')
print(f'  UNSTABLE    : {(mocap_features_df["label"] == "UNSTABLE").sum()}')
print(f'  Skipped     : {skipped}')
print(f'\nFeature columns ({len(mocap_features_df.columns)}):')
print(mocap_features_df.columns.tolist())

mocap_features_df.to_csv(output_path, index=False)
print(f'\n✓ Saved to: {output_path}')